# Pooled co-folding selection: ESMFold2 + OpenFold3

(c) Byron DeLaBarre / The Consulting Biochemist 2026

An attempt to run one `cyp_suite` selection over the union of two generators'
pose sets, rather than selecting within each generator separately.

Consider this a first pass.  Initial results were poor (the LDDT-PLI score was worse than either individual contribution) 
but the idea to find the correct answer by using multiple co-folding approaches/training sets is worth further investigation.  
Some of the selection and weighting may need some work.
Issues:
-EMSFold2 is highly deterministic - unlike other co-folders it does not seem to allow for random seed re-initialization
-as a result, OpenFold3 generates far more structures, swamping the selection process.  This is controllable but I think the 
 ability to make multiple predictions and allow other orthogonal physics based methods to sort them out is a good strategy


Requires: refined, validated pose sets from two `cyp_suite` runs (any
generators), each with `refine.csv` and `validate.csv` in its `base_stage_dir`
and a `harvest_iptm.csv` in its run directory.

In [ ]:
# ============================================================================
# SETUP -- modules, arm split, run registry, and the configs used below.
# ============================================================================
import sys, os, re, shutil, importlib
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import cyp_suite as cs
importlib.reload(cs)
import pandas as pd
import numpy as np
pd.set_option("display.width", 220)

# --- the arm split: heme-iron binders vs non-binders ------------------------
# Naming is forgiving (cmpd_12, cmpd12, CMPD-12, 12, OCNT-2313038 all resolve
# to the same compound) but existence is strict: a compound in neither list,
# in both, or absent from the challenge csv raises here and names itself.
BINDERS = ("cmpd_0", "cmpd_1", "cmpd_5", "cmpd_6", "cmpd_7", "cmpd_9",
           "cmpd_10", "cmpd_11", "cmpd_12", "cmpd_13", "cmpd_17")
NONBINDERS = ("cmpd_2", "cmpd_3", "cmpd_4", "cmpd_8", "cmpd_14", "cmpd_15",
              "cmpd_16", "cmpd_18", "cmpd_19")
ARM_CSV = ""                      # a cmpd,arm table instead of the lists
assert not set(BINDERS) & set(NONBINDERS), "a compound is in both arms"

# --- where the prediction runs live -----------------------------------------
# EDIT: paths and run names are installation-specific.
OF3_ROOT = "/path/to/openfold_work/cyp_predictions"

RUNS = {
    "of_run2": dict(run_name="openadmet20_run2", engine="openfold3",
                    predictions_root=OF3_ROOT),
    "run9":    dict(run_name="esmfold2_cyp3a4_run9", engine="esmfold2"),
}

# --- parameters, each written down once -------------------------------------
SHELL = 3.5
_SH   = f"{SHELL:g}".replace(".", "p")          # 3.5 -> "3p5"

# refinement params: these + the arm map define cfg.tag, hence refine_dir
REFINE_KW = dict(
    arm="mixed", binders=BINDERS, nonbinders=NONBINDERS,
    strict_arms=True, class_csv=ARM_CSV,
    ligand_k=15.0, ligand_tol=0.30, wall_k=1.0e5,
    # a dict, NOT the float 0.26: a float pins both arms and hashes to a
    # different tag than the dict, even though the radii are identical.
    wall_rc_default={"nonbinders": 0.26},
)
# scoring/selection params: deliberately NOT in the tag, so that re-scoring
# does not force a re-refinement. They live in `variant` instead.
SCORE_KW = dict(shell=SHELL, de_tol=0.20, require_pb_pass=True)


def run_config(key, **extra):
    """Config for a RUNS entry, carrying only the refinement params.

    Deliberate: those fix cfg.tag and therefore refine_dir. Pooling reads
    refine.csv and validate.csv, which live in base_stage_dir and are
    variant-independent, so a source run's shell and variant are irrelevant
    here -- the pooled run does its own scoring.
    """
    kw = dict(REFINE_KW); kw.update(extra)
    return cs.RunConfig.from_env(**RUNS[key], **kw)


POOL_SOURCES = {"esm": "run9", "of3": "of_run2"}   # label -> RUNS key
POOL_SEED_OFFSET = 0        # raise only if assemble_pool reports collisions

POOL_SRC = {lbl: run_config(key) for lbl, key in POOL_SOURCES.items()}

POOL = cs.RunConfig.from_env(
    run_name="pooled_" + "_".join(POOL_SOURCES.values()),
    engine="esmfold2",                 # pooled files use this naming convention
    **REFINE_KW, **SCORE_KW,
    variant=f"pooled_shell{_SH}",
)
POOL.iptm_csv = str(POOL.run_dir / "iptm_ranknorm.csv")
POOL.ensure_dirs()
POOL.refine_dir.mkdir(parents=True, exist_ok=True)

for lbl, c in POOL_SRC.items():
    n = len(list(c.refine_dir.glob("*_full.pdb")))
    ok = all(c.manifest(s).is_file() for s in ("refine", "validate"))
    print(f"{lbl:4s} {c.run_name:<30} tag={c.tag}  {n:4d} poses  "
          f"manifests {'OK' if ok else 'MISSING'}")
print(f"\npool  {POOL.refine_dir}")
print(f"iptm  {POOL.iptm_csv}")

In [ ]:
# ============================================================================
# Copy both generators' refined poses into one refine_dir and merge their
# manifests.
#
# Poses are enumerated from refine.csv, not score.csv: the pooled run rescores
# everything, so the source dE values are never used.
#
# ipTM is rank-normalised WITHIN each generator, because confidence values
# from different models are not on a common scale. 1.0 = that generator's
# best pose for the compound, 0.0 = its worst.
# ============================================================================
def assemble_pool(pool, sources, seed_offset=0, dry_run=True):
    """Populate `pool`'s refine_dir and manifests from the source configs.

    sources: {label: RunConfig}. The first label listed gets the seed offset,
    which only matters if the two runs use overlapping seed numbering.
    """
    def remap(stem, off):
        return re.sub(r"_seed_(\d+)_",
                      lambda m: f"_seed_{int(m.group(1)) + off}_", stem)

    plan, refine_rows, val_rows, iptm_rows = [], [], [], []

    for i, (gen, rcfg) in enumerate(sources.items()):
        off = seed_offset if i == 0 else 0
        RF = pd.read_csv(rcfg.manifest("refine"))
        VA = pd.read_csv(rcfg.manifest("validate"))
        IP = cs._load_iptm(rcfg).copy()       # handles both file layouts

        if "skipped" in RF.columns:
            RF = RF[~RF["skipped"].fillna(False).astype(bool)]
        keep = RF[["cmpd", "seed", "sample"]].drop_duplicates()
        print(f"{gen}: {len(keep):4d} refined poses, "
              f"{keep.cmpd.nunique()} compounds")

        rank = IP.groupby("cmpd")["iptm"].rank(ascending=False, method="min")
        n = IP.groupby("cmpd")["iptm"].transform("size")
        IP["iptm_q"] = np.where(n > 1, 1.0 - (rank - 1) / (n - 1), 1.0)

        k = keep.merge(RF, on=["cmpd", "seed", "sample"], how="left")
        for r in k.itertuples():
            for col in ("full", "submission"):      # select needs both
                src = Path(getattr(r, col))
                if not src.is_file():
                    print(f"  [!] missing {col}: {src}"); continue
                plan.append((src, pool.refine_dir / remap(src.name, off)))

        for df, sink in ((k, refine_rows),
                         (keep.merge(VA, on=["cmpd","seed","sample"],
                                     how="left"), val_rows),
                         (keep.merge(IP, on=["cmpd","seed","sample"],
                                     how="left"), iptm_rows)):
            d = df.copy()
            d["seed"] = d["seed"].astype(int) + off
            d["generator"] = gen
            sink.append(d)

    RFM = pd.concat(refine_rows, ignore_index=True)
    VAM = pd.concat(val_rows,    ignore_index=True)
    IPM = pd.concat(iptm_rows,   ignore_index=True)

    dups = RFM.duplicated(["cmpd", "seed", "sample"]).sum()
    print(f"\n{len(plan)} files, {len(RFM)} poses, {dups} key collisions")
    print(RFM.groupby("generator").size().to_string())

    if dups:
        print("  [!] raise seed_offset; refusing"); return None
    if dry_run:
        print("  dry run -- nothing written"); return None

    for s, d in plan:
        shutil.copy2(s, d)
    RFM.to_csv(pool.manifest("refine"), index=False)
    VAM.to_csv(pool.manifest("validate"), index=False)
    IPM[["cmpd", "seed", "sample", "iptm_q"]].rename(
        columns={"iptm_q": "iptm"}).to_csv(pool.iptm_csv, index=False)
    print(f"  {len(list(pool.refine_dir.glob('*_full.pdb')))} *_full.pdb written")
    return RFM


assemble_pool(POOL, POOL_SRC, seed_offset=POOL_SEED_OFFSET, dry_run=True)

In [ ]:
# ============================================================================
# Score the pooled set on one common pocket per compound, then select.
#
# The rescoring is the point. stage_score builds the pocket as the union over
# the poses present, so each source run's dE sits on a different atom set and
# the existing score.csv files cannot simply be concatenated.
# ============================================================================
def pooled_select(pool):
    res = cs.run_all(pool, stages=("score", "select"))
    gens = pd.read_csv(pool.manifest("refine"))[
               ["cmpd", "seed", "sample", "generator"]]
    sc = pd.read_csv(pool.manifest("score")).merge(
             gens, on=["cmpd", "seed", "sample"], how="left")
    sel = res["select"].merge(gens, on=["cmpd", "seed", "sample"], how="left")

    print(f"\n=== picks by generator (shell={pool.shell}) ===")
    print(sel["generator"].value_counts().to_string())

    print("\n=== pocket size / atom-limit check ===")
    chk = (sc.groupby("cmpd")[["n_pocket_res", "n_atoms_complex"]].max()
             .join(sc.groupby("cmpd")["over_max_atoms"].any()))
    print(chk.to_string())
    if chk["over_max_atoms"].any():
        print("  [!] complexes over max_atoms; those energies are suspect")

    print("\n=== dE by generator ===")
    print(sc.groupby("generator")["dE"]
            .agg(["count", "min", "median", "mean"]).round(3))

    print("\n=== selection ===")
    print(sel[["cmpd", "generator", "arm", "seed", "dE", "dE_min",
               "dE_spread", "iptm", "decided_by", "stereo_ok",
               "pb_critical_pass"]].to_string(index=False))
    print(f"\nbest poses -> {pool.best_dir}")
    return sel, sc


selection, scores = pooled_select(POOL)

## Further details & thoughts on the poor result

Submitted to the OpenADMET CYP3A4 leaderboard (LDDT-PLI):

| entry | LDDT-PLI |
|---|---|
| ESMFold2 alone | 0.454 |
| OpenFold3 alone | 0.438 |
| **pooled** | **0.4174** |

The pooled selection scored **below both parents**, and took 16 of 20 picks
from OpenFold3.

## Thoughts on why didn't seem to work

`stage_score` builds the scoring pocket as the union over the poses in the
set. Pooling therefore does not merge two candidate lists under a fixed
criterion — it *changes the criterion* for every pose.

ESMFold2 and OpenFold3 place ligands in substantially different regions of
the CYP3A4 pocket: comparing the two single-generator submissions after
protein superposition gives a median ligand-ligand RMSD of ~8 Å, with the
contact-residue sets overlapping by only ~0.31 (Jaccard). The union pocket
therefore spans both regions and is far more diffuse than either generator's
own. Against a diffuse pocket, dE rewards whichever pose contacts more
residues that only some poses touch — which is not the same as rewarding the
correct pose.

**dE is comparable only within a pose set whose union pocket is coherent.**
That holds comfortably within a single run and fails across generators that
place ligands differently.

This needs further investigation - dE calculation was created in the accompanying cyp_suite_v8.py and perhaps not
vetted as closely as it should have been.  It works well within a single prediction, but orbmol computed energy should be objective and
not tied to the co-folding method! 


## Corrections tried 

Two adjustments were tried and both made things worse, for reasons worth
recording:

- **Equalising pool sizes.** OpenFold3 contributed ~24 poses per compound
  against ESMFold2's ~5, so argmin over the pooled set favours the larger
  pool. But subsampling to equal size left the split unchanged at 17/3, and
  the premise is questionable anyway: if one generator searched more widely
  and found a lower-energy pose, that is a result, not a bias.
- **Median-centring dE within (compound, generator).** This made the split
  worse (18/2). ESMFold2's five poses straddle their own median — the best
  sits only 0.11 eV below it — whereas OpenFold3's best is 0.36 eV below its
  median, being the tail of a much wider distribution. Centring on the full
  set while selecting from the top substitutes a spread artifact for a
  location artifact. Centring on the subsample instead flips the bias the
  other way.

